In [0]:
# ════════════════════════════════════════════
# Workshop Configuration — no edits needed
# ════════════════════════════════════════════
import re

# Read the current user directly from the Databricks workspace context
_raw_user   = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
USERNAME    = re.sub(r'[^a-zA-Z0-9]', '_', _raw_user.split("@")[0])

CATALOG     = "workshop"         # shared catalog, pre-configured by facilitator
SCHEMA      = USERNAME           # your personal schema inside the catalog

# MLflow experiment path — lives in your personal workspace folder
EXPERIMENT_PATH = f"/Users/{_raw_user}/gdm_yield_{USERNAME}"

print(f"📍 Logged in as  : {_raw_user}")
print(f"📍 USERNAME       : {USERNAME}")
print(f"📍 Your namespace : {CATALOG}.{SCHEMA}")
print(f"📍 Your table     : {CATALOG}.{SCHEMA}.yield_trials")
print(f"📍 Your model     : gdm-yield-model-{USERNAME}")
print(f"📍 MLflow path    : {EXPERIMENT_PATH}")

In [0]:
# Create your personal schema (run once)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
print(f"✅ Schema ready: {CATALOG}.{SCHEMA}")

In [0]:
# Install Kaggle CLI and download the crop yield dataset
#-ñ Note: Requires Kaggle API credenti1
# als at ~/.kaggle/kaggle.json
# Get yours from: https://www.kaggle.com/settings -> Create New API Token

%pip install kaggle --quiet

import os
import zipfile

# Set kaggle dataset
dataset_name = "patelris/crop-yield-prediction-dataset"

# Download the dataset
print("📥 Downloading dataset from Kaggle...")
!kaggle datasets download -d {dataset_name} -p /tmp --force

# Extract the ZIP file
zip_path = "/tmp/crop-yield-prediction-dataset.zip"
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("/tmp/crop_yield/")

print("✅ Dataset downloaded and extracted to /tmp/crop_yield/")
print("📄 Files available:")
for file in os.listdir("/tmp/crop_yield/"):
    print(f"   - {file}")

In [0]:
# Load the yield_df.csv into a Spark DataFrame
# On serverless: use pandas to read local file, then convert to Spark
import pandas as pd

csv_path = "/tmp/crop_yield/yield_df.csv"

# Read with pandas first (can access local filesystem)
pandas_df = pd.read_csv(csv_path)

# Convert pandas DataFrame to Spark DataFrame
df_yield = spark.createDataFrame(pandas_df)

print(f"✅ Loaded yield_df.csv into DataFrame 'df_yield'")
print(f"   Columns: {len(df_yield.columns)}")
print(f"   Rows: {df_yield.count()}")

In [0]:
display(df_yield)

In [0]:
# Display schema (column names and types)
print("📄 Schema:")
print("=" * 60)
df_yield.printSchema()

print("\n🔢 Total Row Count:")
print("=" * 60)
row_count = df_yield.count()
print(f"{row_count:,} rows")

print("\n📊 Sample Rows (first 5):")
print("=" * 60)
display(df_yield.limit(5))

In [0]:
# Show unique values in the "Item" column (crop types)
print("🌾 Unique Crop Types (Item column):")
print("=" * 60)
crop_types = df_yield.select("Item").distinct().orderBy("Item").collect()
for row in crop_types:
    print(f"   • {row['Item']}")

print(f"\nTotal unique crops: {len(crop_types)}")

print("\n\n🌎 Countries containing 'Argentina' or 'Brazil':")
print("=" * 60)
filtered_countries = df_yield.filter(
    (df_yield["Area"].contains("Argentina")) | (df_yield["Area"].contains("Brazil"))
).select("Area").distinct().orderBy("Area").collect()

for row in filtered_countries:
    print(f"   • {row['Area']}")

print(f"\nTotal matching countries: {len(filtered_countries)}")

In [0]:
# Save the DataFrame as a Delta table
# First, clean up column names (Delta doesn't allow special characters like : and /)
table_name = f"{CATALOG}.{SCHEMA}.yield_trials"

print(f"💾 Preparing DataFrame for Delta table...")
df_clean = df_yield \
    .withColumnRenamed("Unnamed: 0", "id") \
    .withColumnRenamed("hg/ha_yield", "hg_ha_yield")

print(f"💾 Saving DataFrame to: {table_name}")
df_clean.write.format("delta").mode("overwrite").saveAsTable(table_name)
print(f"✅ Table saved successfully!")

In [0]:
%sql
-- Verify the number of rows in the table
SELECT COUNT(*) as total_rows
FROM workshop.yltabord.yield_trials

In [0]:
%sql
-- View sample rows from the table
SELECT *
FROM workshop.yltabord.yield_trials
LIMIT 5

In [0]:
# Load the Delta table into a pandas DataFrame
table_name = f"{CATALOG}.{SCHEMA}.yield_trials"

print(f"📥 Loading table: {table_name}")
df = spark.table(table_name).toPandas()

print(f"✅ Loaded {len(df):,} rows into pandas DataFrame")
print(f"\nColumns: {list(df.columns)}")
print(f"\nDataFrame shape: {df.shape}")
df.head()

In [0]:
# Install XGBoost for the XGBRegressor model
%pip install xgboost --quiet

In [0]:
# Build sklearn Pipeline that encapsulates ALL preprocessing
#!pip install xgboost
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from xgboost import XGBRegressor

# Define feature columns and target
feature_cols = ["Item", "Area", "average_rain_fall_mm_per_year", "pesticides_tonnes", "avg_temp", "Year"]
target_col = "hg_ha_yield"

# Separate features and target
X = df[feature_cols]
y = df[target_col]

print(f"✅ Features shape: {X.shape}")
print(f"✅ Target shape: {y.shape}")

# Build ColumnTransformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), ["Item", "Area"]),
        ("numerical", "passthrough", ["average_rain_fall_mm_per_year", "pesticides_tonnes", "avg_temp", "Year"])
    ]
)

# Build the full pipeline
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", XGBRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42))
])

print(f"\n✅ Pipeline created:")
print(pipeline)

In [0]:
# Split data 80/20 train/test with random_state=42
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42
)

print("📊 Train/Test Split Results:")
print("=" * 60)
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape:  {y_test.shape}")

print(f"\n✅ Train set: {len(X_train):,} samples ({len(X_train)/len(X)*100:.1f}%)")
print(f"✅ Test set:  {len(X_test):,} samples ({len(X_test)/len(X)*100:.1f}%)")

print(f"\n📋 Sample from X_train (raw strings preserved):")
print(X_train.head())

In [0]:
# Train the pipeline and log to MLflow
import mlflow
from mlflow.models import infer_signature
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# Set MLflow experiment to user's workspace folder
mlflow.set_experiment(EXPERIMENT_PATH)
print(f"🧪 MLflow experiment set to: {EXPERIMENT_PATH}")

# Train the pipeline
print(f"\n🚀 Training pipeline on {len(X_train):,} samples...")
pipeline.fit(X_train, y_train)
print("✅ Pipeline training complete!")

# Make predictions on test set
y_pred = pipeline.predict(X_test)

# Calculate evaluation metrics
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"\n📊 Evaluation Metrics:")
print("=" * 60)
print(f"RMSE: {rmse:,.2f}")
print(f"R²:   {r2:.4f}")

# Infer model signature (required for Unity Catalog)
signature = infer_signature(X_train, pipeline.predict(X_train))

# Log model and metrics to MLflow
with mlflow.start_run() as run:
    # Log parameters
    mlflow.log_params({
        "n_estimators": 100,
        "max_depth": 5,
        "learning_rate": 0.1
    })
    
    # Log metrics
    mlflow.log_metrics({
        "rmse": rmse,
        "r2": r2
    })
    
    # Log the full pipeline (includes OrdinalEncoder + XGBoost)
    registered_model_name = f"{CATALOG}.{SCHEMA}.gdm-yield-model-{USERNAME}"
    mlflow.sklearn.log_model(
        sk_model=pipeline,
        artifact_path="gdm_yield_pipeline",
        signature=signature,
        input_example=X_train[:5],
        registered_model_name=registered_model_name
    )
    
    print(f"\n✅ Model logged to MLflow!")
    print(f"   Run ID: {run.info.run_id}")
    print(f"   Registered model: {registered_model_name}")

In [0]:
# Create horizontal bar chart of feature importances
import plotly.graph_objects as go

# Extract feature importances from the trained XGBoost model
feature_importances = pipeline.named_steps["regressor"].feature_importances_

# Feature names (in ColumnTransformer order)
feature_names = [
    "Crop Type",
    "Country",
    "Rainfall (mm/yr)",
    "Pesticides (t)",
    "Avg Temp (°C)",
    "Year"
]

# Sort by importance (descending)
sorted_idx = np.argsort(feature_importances)[::-1]
sorted_importances = feature_importances[sorted_idx]
sorted_names = [feature_names[i] for i in sorted_idx]

# Create horizontal bar chart
fig = go.Figure(go.Bar(
    x=sorted_importances,
    y=sorted_names,
    orientation='h',
    marker=dict(color='steelblue')
))

fig.update_layout(
    title="Feature Importance (XGBoost)",
    xaxis_title="Importance Score",
    yaxis_title="Feature",
    height=400,
    yaxis=dict(autorange="reversed")  # Highest importance at top
)

fig.show()
print("✅ Feature importance chart displayed")

In [0]:
# Create scatter plot: predicted vs actual yield
import plotly.express as px

# Create DataFrame for plotting
results_df = pd.DataFrame({
    'Actual': y_test,
    'Predicted': y_pred
})

# Create scatter plot
fig = px.scatter(
    results_df,
    x='Actual',
    y='Predicted',
    opacity=0.6,
    labels={'Actual': 'Actual Yield (hg/ha)', 'Predicted': 'Predicted Yield (hg/ha)'},
    title='Predicted vs Actual Crop Yield'
)

# Add perfect prediction line (diagonal)
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())

fig.add_scatter(
    x=[min_val, max_val],
    y=[min_val, max_val],
    mode='lines',
    line=dict(color='red', dash='dash'),
    name='Perfect Prediction'
)

fig.update_layout(
    height=500,
    width=600
)

fig.show()
print(f"✅ Predictions scatter plot displayed")
print(f"   Total test samples: {len(y_test):,}")